> **Repository copy:** code only, with execution outputs removed. Run only in an authorized All of Us Workbench. Generated tables, audits, plots, HTML exports, and individual-level intermediate files stay in that environment. See `README.md` and `docs/provenance.md` for setup and version details.


# Documented Cardiovascular Testing and Stroke After Retinal Artery Occlusion and Transient Monocular Vision Loss
**All of Us Registered Tier, CDR v9 (R2025Q4R6). Cases-only design.**

**Primary question:** after a first retinal artery occlusion (CRAO/BRAO), how often is cardiovascular testing documented, and how does this compare with transient monocular vision loss (TMVL), after adjustment for clinical history, healthcare use, and the care setting of the diagnosis?


**Secondary:** cumulative incidence of new stroke after each condition, and a direct adjusted RAO-vs-TMVL comparison.

**How to run**
1. Put the reviewed `procedure_concepts.csv` in the `definitions/` folder (if the file is missing, the notebook writes a fresh candidate list and **stops on purpose** so it can be reviewed).
2. *Kernel → Restart Kernel and Run All Cells*.
3. The last cell copies the exact definitions file used into `results/` and prints its checksum, so outputs can be tied to the definitions.


**Design decisions**
- RAO primary definition = CRAO and BRAO concepts only. Partial/unspecified occlusion, thrombosis, and cholesterol embolus are used only in a broad-definition sensitivity analysis.
- Exposure = earliest qualifying event (RAO wins same-day ties). TMVL followed later by RAO stays in the TMVL group.
- Eligibility uses baseline information only: age ≥18 and ≥1 year of prior records. No requirement for future records.
- Testing is reported as *documented* testing. Each test excludes people with the same test in the prior 180 days; rhythm monitoring excludes known AF; statin initiation is among the statin-naive. The 90-day testing analyses include people observable for 90 days (a record on or after day 90, or death within 90 days); a sensitivity analysis drops this requirement.
- No matched controls (Registered Tier dates are shifted separately per participant, so calendar alignment of controls is unreliable). Diagnosis periods use shifted dates and are approximate; a sensitivity analysis excludes the shifted years just before each cut point.
- Disclosure control: counts ≤20 suppressed, including percentages whose complement is ≤20 and categories that could be recovered by subtraction.

## 0. Setup

In [ ]:
import os, re, warnings
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from google.cloud import bigquery

DEFINITIONS_FROZEN = True      # procedure list already reviewed; set False only to force a stop for re-review

DS = os.environ.get("ALL_OF_US_CDR", "").strip()
if not re.fullmatch(r"[A-Za-z0-9_-]+\.[A-Za-z0-9_]+", DS):
    raise ValueError("Set ALL_OF_US_CDR to the authorized project.dataset for Registered Tier CDR v9 before running.")
client = bigquery.Client()
os.makedirs("results", exist_ok=True)
os.makedirs("definitions", exist_ok=True)
pd.set_option("display.max_rows", 300, "display.max_colwidth", 110, "display.width", 200)

def bq(sql):
    """Run a query; convert BigQuery's nullable integer/boolean columns to plain NumPy types when possible."""
    df = client.query(sql).to_dataframe()
    for c in df.columns:
        dt = str(df[c].dtype)
        if dt in ("Int64", "Int32", "Int16", "Int8") and not df[c].isna().any():
            df[c] = df[c].astype("int64")
        elif dt in ("Int64", "Int32", "Int16", "Int8"):
            df[c] = df[c].astype("float64")
        elif dt == "boolean":
            df[c] = df[c].fillna(False).astype(bool)
    return df

ids = lambda l: ",".join(str(int(x)) for x in l)

MIN_CELL = 20
def ok_count(x, n=None):
    return x > MIN_CELL and (n is None or n - x > MIN_CELL)
def n_str(n):
    return f"{n:,}" if n > MIN_CELL else "≤20"
def pct_str(x, n):
    return f"{100 * x / n:.1f}%" if ok_count(x, n) else "suppressed"
def cnt_str(x, n):
    return f"{x:,} ({100 * x / n:.1f}%)" if ok_count(x, n) else "suppressed"
def fmt(est, lo, hi, d=2):
    return f"{est:.{d}f} ({lo:.{d}f}–{hi:.{d}f})" if pd.notna(est) else "not estimated"

def partition_rows(d, col, levels, label):
    """Categories that sum to N: if exactly one is suppressed, also suppress the next smallest."""
    n = len(d)
    xs = {l: int((d[col] == l).sum()) for l in levels}
    sup = {l: not ok_count(xs[l], n) for l in levels}
    if sum(sup.values()) == 1:
        others = [l for l in levels if not sup[l]]
        if others:
            sup[min(others, key=lambda l: xs[l])] = True
    return {f"{label}: {l}": ("suppressed" if sup[l] else f"{xs[l]:,} ({100 * xs[l] / n:.1f}%)") for l in levels}

def save(df, name, index=True):
    df.to_csv(f"results/{name}.csv", index=index)
    print(f"--- {name} (saved to results/{name}.csv) ---")
    display(df)

FIT_LOG = []   # every model attempted, with success/failure

## 1. Exposure definitions and diagnosis code audit

In [ ]:
CRAO = [437540, 37207895, 37207955, 37310623]            # central RAO, right/left/bilateral
BRAO = [36713329, 36713330, 3657106]                     # branch RAO, left/right/bilateral
RAO_STRICT = CRAO + BRAO
RAO_OTHER = [437847, 3656884, 3656968, 4334245, 4096157, 3657144]   # partial/unspecified/thrombosis/embolus: broad sensitivity only
TMVL = [4338523, 36712812, 36712813, 3656941, 3656857]   # amaurosis fugax (+ laterality), transient retinal artery occlusion
ALL_EXPO = RAO_STRICT + RAO_OTHER + TMVL

names = bq(f"""SELECT concept_id, concept_name, vocabulary_id, standard_concept
               FROM `{DS}.concept` WHERE concept_id IN ({ids(ALL_EXPO)})""")
names["set"] = names.concept_id.map(lambda c: "RAO strict (CRAO/BRAO)" if c in RAO_STRICT
                                    else "RAO other (broad only)" if c in RAO_OTHER else "TMVL")
names.sort_values(["set", "concept_id"]).to_csv("definitions/exposure_concepts.csv", index=False)
display(names.sort_values(["set", "concept_id"]))

audit = bq(f"""SELECT condition_concept_id AS concept_id, condition_source_value AS source_code,
                      COUNT(DISTINCT person_id) AS people
               FROM `{DS}.condition_occurrence`
               WHERE condition_concept_id IN ({ids(ALL_EXPO)}) GROUP BY 1, 2""")
audit = audit.merge(names[["concept_id", "concept_name", "set"]], on="concept_id")
audit.to_csv("definitions/diagnosis_source_audit_INTERNAL.csv", index=False)   # contains small counts: keep inside the Workbench
print("Source codes behind each exposure concept (rows with >20 people):")
display(audit[audit.people > MIN_CELL].sort_values(["set", "people"], ascending=[True, False])
        [["set", "concept_name", "source_code", "people"]])
print(f"Rows hidden (≤20 people): {(audit.people <= MIN_CELL).sum()}")

## 2. Outcome and covariate definitions

In [ ]:
HIST = r"\bold\b|history|sequela|late effect|demand ischemia"     # counts only as prior history
OUT_ANC = [313217, 443454, 381316, 4329847]      # AF; cerebral infarction; CVA; MI (with descendants)
COMORB = {"Hypertension": 316866, "Diabetes": 201820, "Hyperlipidemia": 432867,
          "Coronary artery disease": 317576, "Heart failure": 316139, "Chronic kidney disease": 46271022}
print(bq(f"SELECT concept_id, concept_name FROM `{DS}.concept` "
         f"WHERE concept_id IN ({ids(OUT_ANC + list(COMORB.values()))})"))

BASE_CLIN = list(COMORB) + ["Prior AF", "Prior stroke", "Prior MI"]
RACE_LEVELS = ["White (NH)", "Black (NH)", "Hispanic", "Other/Unknown"]
SETTING_LEVELS = ["ED/inpatient", "Outpatient", "Unknown"]
ERA_LEVELS = ["2016 or earlier", "2017-2020", "2021+"]

def get_events(pids):
    return bq(f"""
    SELECT DISTINCT co.person_id, co.condition_start_date AS d, c.concept_name,
      CASE WHEN ca.ancestor_concept_id = 313217 THEN 'AF'
           WHEN ca.ancestor_concept_id IN (443454, 381316) THEN 'Stroke'
           ELSE 'MI' END AS outcome
    FROM `{DS}.condition_occurrence` co
    JOIN `{DS}.concept_ancestor` ca ON co.condition_concept_id = ca.descendant_concept_id
    JOIN `{DS}.concept` c ON c.concept_id = co.condition_concept_id
    WHERE ca.ancestor_concept_id IN ({ids(OUT_ANC)})
      AND co.condition_concept_id NOT IN ({ids(ALL_EXPO)})
      AND co.person_id IN ({ids(pids)})""")

def race_eth(r):
    """Non-Hispanic White/Black only when ethnicity is explicitly 'Not Hispanic or Latino'."""
    eth = r.ethnicity.lower() if isinstance(r.ethnicity, str) else ""
    if eth.startswith("hispanic"):
        return "Hispanic"
    if eth.startswith("not hispanic"):
        return {"White": "White (NH)", "Black or African American": "Black (NH)"}.get(r.race, "Other/Unknown")
    return "Other/Unknown"      # other race, multiple, or ethnicity missing/unknown

## 3. Cohorts (earliest qualifying event and baseline only eligibility)

In [ ]:
ev_x = bq(f"""SELECT DISTINCT person_id, condition_start_date AS d, condition_concept_id AS cid,
                     visit_occurrence_id AS vid
              FROM `{DS}.condition_occurrence` WHERE condition_concept_id IN ({ids(ALL_EXPO)})""")
ev_x["d"] = pd.to_datetime(ev_x.d)
ev_x["cat"] = np.select([ev_x.cid.isin(RAO_STRICT).to_numpy(dtype=bool),
                         ev_x.cid.isin(TMVL).to_numpy(dtype=bool)], ["RAO", "TMVL"], "RAO_OTHER")
expo_ids = ev_x.person_id.unique()

pers = bq(f"""SELECT p.person_id, p.year_of_birth, r.concept_name AS race, e.concept_name AS ethnicity,
                     s.concept_name AS sex
              FROM `{DS}.person` p
              LEFT JOIN `{DS}.concept` r ON r.concept_id = p.race_concept_id
              LEFT JOIN `{DS}.concept` e ON e.concept_id = p.ethnicity_concept_id
              LEFT JOIN `{DS}.concept` s ON s.concept_id = p.sex_at_birth_concept_id
              WHERE p.person_id IN ({ids(expo_ids)})""")
pers["race_eth"] = pers.apply(race_eth, axis=1)
pers["sex"] = pers.sex.where(pers.sex.isin(["Female", "Male"]), "Other/Unknown")

vis = bq(f"""SELECT DISTINCT v.person_id, v.visit_occurrence_id AS vid, v.visit_start_date AS vd,
                    v.visit_end_date AS ve, c.concept_name AS vtype
             FROM `{DS}.visit_occurrence` v LEFT JOIN `{DS}.concept` c ON c.concept_id = v.visit_concept_id
             WHERE v.person_id IN ({ids(expo_ids)})""")
vis["vd"] = pd.to_datetime(vis.vd)
vis["ve"] = pd.to_datetime(vis.ve).fillna(vis.vd)
vis.loc[vis.ve < vis.vd, "ve"] = vis.vd
ACUTE_PAT = r"emergency|inpatient|observation|intensive care"
OUTPT_PAT = r"outpatient|office|telehealth|telemedicine|ambulatory"
vis["acute"] = vis.vtype.str.contains(ACUTE_PAT, case=False, na=False).to_numpy(dtype=bool)
vis["outpt"] = (~vis.acute & vis.vtype.str.contains(OUTPT_PAT, case=False, na=False)).to_numpy(dtype=bool)
vis["vclass"] = np.select([vis.acute, vis.outpt], ["ED/inpatient", "Outpatient"], "Unknown/other")
_vt = vis.groupby(["vtype", "vclass"], dropna=False).person_id.nunique().reset_index(name="people")
_vt["vtype"] = _vt.vtype.fillna("(missing)")
_vt.to_csv("definitions/visit_type_classification_INTERNAL.csv", index=False)
print("Visit types and their care-setting class (types with >20 people):")
display(_vt[_vt.people > MIN_CELL].sort_values("people", ascending=False))
print(f"Visit types hidden (≤20 people): {(_vt.people <= MIN_CELL).sum()}")
obs = (vis.groupby("person_id").agg(first_visit=("vd", "min"), last_visit=("ve", "max")).reset_index())   # last = latest visit END
dth = bq(f"""SELECT person_id, MIN(death_date) AS death_date FROM `{DS}.death`
             WHERE person_id IN ({ids(expo_ids)}) GROUP BY 1""")
dth["death_date"] = pd.to_datetime(dth.death_date)

def build_exposure(rao_cats):
    f_rao = ev_x[ev_x.cat.isin(rao_cats)].groupby("person_id").d.min().rename("first_rao")
    f_tmvl = ev_x[ev_x.cat == "TMVL"].groupby("person_id").d.min().rename("first_tmvl")
    df = pd.concat([f_rao, f_tmvl], axis=1)
    df.index.name = "person_id"
    df = df.reset_index()
    df["index_date"] = df[["first_rao", "first_tmvl"]].min(axis=1)
    rao_first = df.first_rao.notna() & (df.first_rao <= df.first_tmvl.fillna(pd.Timestamp.max))
    df["grp"] = np.where(rao_first, "RAO", "TMVL")
    df["later_rao_after_tmvl"] = (df.grp == "TMVL") & df.first_rao.notna()
    n_dates = ev_x[ev_x.cat.isin(rao_cats)].groupby("person_id").d.nunique()
    df["confirmed"] = np.where(df.grp == "RAO", df.person_id.map(n_dates).fillna(0) >= 2, True)
    return df

def make_cohort(rao_cats):
    df = (build_exposure(rao_cats).merge(pers, on="person_id", how="left")
          .merge(obs, on="person_id", how="left").merge(dth, on="person_id", how="left"))
    df["age"] = (df.index_date.dt.year - df.year_of_birth).astype(float)
    df["lookback_days"] = (df.index_date - df.first_visit).dt.days
    keep = (df.age >= 18) & (df.lookback_days >= 365)
    flow = pd.DataFrame({g: {
        "First qualifying diagnosis": n_str(int((df.grp == g).sum())),
        "Analytic cohort (age ≥18, ≥1 yr prior records)": n_str(int((keep & (df.grp == g)).sum())),
        "TMVL later coded as RAO (kept as TMVL)": (n_str(int((keep & df.later_rao_after_tmvl).sum()))
                                                    if g == "TMVL" else "—")}
        for g in ["RAO", "TMVL"]})
    coh = df[keep].copy()
    coh["obs_end_day"] = (coh.last_visit - coh.index_date).dt.days.clip(lower=0)
    coh["death_day"] = (coh.death_date - coh.index_date).dt.days
    coh["era"] = pd.cut(coh.index_date.dt.year, [0, 2016, 2020, 2100], labels=ERA_LEVELS).astype(str)
    coh["boundary_year"] = coh.index_date.dt.year.isin([2016, 2020])   # shifted year may belong to the next period
    return coh.sort_values(["grp", "person_id"]).reset_index(drop=True), flow

coh_main, flow_main = make_cohort(["RAO"])
coh_broad, flow_broad = make_cohort(["RAO", "RAO_OTHER"])
for row in flow_broad.index:        # TMVL differs between definitions only by people reclassified; hide small differences
    a, b = flow_main.loc[row, "TMVL"], flow_broad.loc[row, "TMVL"]
    try:
        if 0 < abs(int(str(a).replace(",", "")) - int(str(b).replace(",", ""))) <= MIN_CELL:
            flow_broad.loc[row, "TMVL"] = "suppressed (differs from primary by ≤20)"
    except ValueError:
        pass
save(flow_main, "T0_cohort_flow_primary")
save(flow_broad, "S0_cohort_flow_broad_definition")

## 4. Baseline covariates, prior year healthcare use, and care setting of the diagnosis

In [ ]:
def add_covariates(coh):
    coh = coh.copy()
    pids = coh.person_id.unique()
    cm = bq(f"""SELECT co.person_id, ca.ancestor_concept_id AS anc, MIN(co.condition_start_date) AS first_d
                FROM `{DS}.condition_occurrence` co
                JOIN `{DS}.concept_ancestor` ca ON co.condition_concept_id = ca.descendant_concept_id
                WHERE ca.ancestor_concept_id IN ({ids(COMORB.values())}) AND co.person_id IN ({ids(pids)})
                GROUP BY 1, 2""")
    cm["first_d"] = pd.to_datetime(cm.first_d)
    for name, cid in COMORB.items():
        fd = coh.person_id.map(cm[cm.anc == cid].set_index("person_id").first_d)
        coh[name] = (fd < coh.index_date).fillna(False).astype(bool)

    ev = get_events(pids).merge(coh[["person_id", "index_date"]], on="person_id")
    ev["days"] = (pd.to_datetime(ev.d) - ev.index_date).dt.days
    ev["is_hist"] = ev.concept_name.str.contains(HIST, case=False)
    for o, lab in [("AF", "Prior AF"), ("Stroke", "Prior stroke"), ("MI", "Prior MI")]:
        coh[lab] = coh.person_id.isin(ev.loc[(ev.outcome == o) & (ev.days < 0), "person_id"])
    new_stroke = ev[(ev.outcome == "Stroke") & ~ev.is_hist & (ev.days >= 0)].groupby("person_id").days.min()
    coh["stroke_day"] = coh.person_id.map(new_stroke)

    v = vis.merge(coh[["person_id", "index_date"]], on="person_id")
    v["days"] = (v.vd - v.index_date).dt.days
    coh["prior_visit_days"] = coh.person_id.map(
        v[v.days.between(-365, -1)].groupby("person_id").vd.nunique()).fillna(0)
    # Care setting of the index diagnosis (v2.3):
    # ED/inpatient: the encounter linked to an index-date diagnosis record is acute, OR an acute encounter spans
    #               the index date (start-1 to end+1).
    # Outpatient:   otherwise, the linked encounter is a recognized outpatient type, OR a recognized outpatient
    #               encounter starts within ±1 day. Unrecognized/missing visit types never count as outpatient.
    # Unknown:      everything else.
    rec = ev_x.merge(coh[["person_id", "index_date"]], on="person_id")
    rec = rec[(rec.d == rec.index_date) & rec.vid.notna()].copy()
    rec["vid"] = rec.vid.astype("int64")
    linked = rec.merge(vis[["vid", "acute", "outpt"]], on="vid", how="inner")
    linked_acute = set(linked.loc[linked.acute, "person_id"])
    linked_outpt = set(linked.loc[linked.outpt, "person_id"])
    one = pd.Timedelta(days=1)
    span_acute = set(v.loc[v.acute & (v.vd <= v.index_date + one) & (v.ve >= v.index_date - one), "person_id"])
    near_outpt = set(v.loc[v.outpt & v.days.between(-1, 1), "person_id"])
    pid = coh.person_id
    coh["setting"] = np.select(
        [pid.isin(linked_acute | span_acute).to_numpy(dtype=bool),
         pid.isin(linked_outpt | near_outpt).to_numpy(dtype=bool)],
        ["ED/inpatient", "Outpatient"], "Unknown")
    return coh

coh_main = add_covariates(coh_main)
coh_broad = add_covariates(coh_broad)

def baseline_table(coh):
    cols = {}
    for g in ["RAO", "TMVL"]:
        d = coh[coh.grp == g]
        n = len(d)
        c = {"N": n_str(n), "Age, mean (SD)": f"{d.age.mean():.1f} ({d.age.std():.1f})",
             "Female": cnt_str(int((d.sex == "Female").sum()), n)}
        c.update(partition_rows(d, "race_eth", RACE_LEVELS, "Race/ethnicity"))
        for v in BASE_CLIN:
            c[v] = cnt_str(int(d[v].sum()), n)
        q = d.prior_visit_days.quantile([.25, .5, .75])
        c["Visit days in prior year, median (IQR)"] = f"{q[.5]:.0f} ({q[.25]:.0f}–{q[.75]:.0f})"
        c.update(partition_rows(d, "setting", SETTING_LEVELS, "Care setting of diagnosis"))
        c.update(partition_rows(d, "era", ERA_LEVELS, "Diagnosis period (shifted dates)"))
        cols[g] = c
    return pd.DataFrame(cols)

save(baseline_table(coh_main), "T1_baseline_RAO_vs_TMVL")

## 5. Procedure definitions 

In [ ]:
TEST_PATTERNS = {
    "echo":    r"echocardiog",
    "rhythm":  r"holter|external electrocardiographic recording|event record|cardiac rhythm monitor|cardiovascular telemetry|mobile cardiac|loop recorder|ambulatory (?:ecg|ekg|electrocardiog)",
    "carotid": r"carotid.*(?:duplex|ultraso|doppler|angiog|arteriog|imaging)|extracranial arter|(?:angiograph|arteriograph).*neck|neck.*(?:angiograph|arteriograph)",
    "brain":   r"(?:magnetic resonance|\bmri\b|computed tomograph|\bct\b).*(?:brain|head)|(?:brain|head).*(?:magnetic resonance|\bmri\b|computed tomograph|\bct\b)",
}
TEST_LABELS = {"echo": "Echocardiography (excl. stress)", "rhythm": "Ambulatory rhythm monitoring",
               "carotid": "Carotid/neck vessel imaging", "brain": "Brain imaging"}
all_ids = np.union1d(coh_main.person_id.unique(), coh_broad.person_id.unique())

PROC_FILE = "definitions/procedure_concepts.csv"
fresh = not os.path.exists(PROC_FILE)
if fresh:
    pc = bq(f"""
    WITH p AS (
      SELECT person_id, procedure_concept_id AS cid FROM `{DS}.procedure_occurrence` WHERE person_id IN ({ids(all_ids)})
      UNION ALL
      SELECT person_id, procedure_source_concept_id FROM `{DS}.procedure_occurrence` WHERE person_id IN ({ids(all_ids)}))
    SELECT c.concept_id, c.concept_name, c.vocabulary_id, c.concept_code, COUNT(DISTINCT p.person_id) AS people
    FROM p JOIN `{DS}.concept` c ON c.concept_id = p.cid
    GROUP BY 1, 2, 3, 4""")
    rows = []
    for k, pat in TEST_PATTERNS.items():
        m = pc[pc.concept_name.str.contains(pat, case=False, regex=True)]
        if k == "echo":
            m = m[~m.concept_name.str.contains("stress", case=False)]
        rows.append(m.assign(test=k, include=True))
    cand = pd.concat(rows)
    cand["people"] = cand.people.where(cand.people > MIN_CELL)        # small counts not stored
    cand.sort_values(["test", "people"], ascending=[True, False]).to_csv(PROC_FILE, index=False)
    print(f"Wrote {len(cand)} candidate procedure concepts to {PROC_FILE}")

if fresh or not DEFINITIONS_FROZEN:
    raise RuntimeError("STOP (intended): review definitions/procedure_concepts.csv. Set include=False for rows "
                       "that are not the intended test, save, then set DEFINITIONS_FROZEN = True in the setup "
                       "cell and run all cells again.")

proc_all = pd.read_csv(PROC_FILE)
proc_all["include"] = proc_all.include.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
# Findings ("... abnormal") do not establish that/when a test was performed
finding = proc_all.concept_name.str.contains(r"\babnormal\b", case=False)
proc_all.loc[finding, "include"] = False
# Angiograms that image the neck vessels also count as neck vessel imaging
neck_angio = proc_all.include & proc_all.concept_name.str.contains(r"angiogra\w*.*neck", case=False)
add = proc_all[neck_angio & (proc_all.test != "carotid")]
add = add[~add.concept_id.isin(proc_all.loc[proc_all.test == "carotid", "concept_id"])].assign(test="carotid")
proc_all = pd.concat([proc_all, add], ignore_index=True)
proc_def = proc_all[proc_all.include]
proc_excl = proc_all[~proc_all.include]
print(f"Findings dropped: {int(finding.sum())}; head-and-neck angiograms added to carotid: {len(add)}")
print("Frozen procedure concepts per test:")
print(proc_def.groupby("test").concept_id.nunique().to_string())

## 6. Documented testing and statin initiation

In [ ]:
tp = bq(f"""SELECT DISTINCT person_id, procedure_date AS d,
                   procedure_concept_id AS c1, procedure_source_concept_id AS c2
            FROM `{DS}.procedure_occurrence`
            WHERE person_id IN ({ids(all_ids)})
              AND (procedure_concept_id IN ({ids(proc_def.concept_id.unique())})
                   OR procedure_source_concept_id IN ({ids(proc_def.concept_id.unique())}))""")
tp["d"] = pd.to_datetime(tp.d)
for k in TEST_PATTERNS:
    kid = set(proc_def.loc[proc_def.test == k, "concept_id"])
    kex = set(proc_excl.loc[proc_excl.test == k, "concept_id"])
    hit = tp.c1.isin(kid) | tp.c2.isin(kid)
    blocked = tp.c1.isin(kex) | tp.c2.isin(kex)     # excluded source code cannot re-enter via its standard concept
    tp[k] = (hit & ~blocked).to_numpy(dtype=bool)

statin_ing = bq(f"""SELECT concept_id FROM `{DS}.concept`
    WHERE vocabulary_id = 'RxNorm' AND concept_class_id = 'Ingredient'
      AND LOWER(concept_name) IN ('atorvastatin','rosuvastatin','simvastatin','pravastatin',
                                  'lovastatin','fluvastatin','pitavastatin')""").concept_id.tolist()
print(f"Statin ingredients found: {len(statin_ing)} (expected 7)")
st = bq(f"""SELECT DISTINCT de.person_id, de.drug_exposure_start_date AS d
            FROM `{DS}.drug_exposure` de
            JOIN `{DS}.concept_ancestor` ca ON de.drug_concept_id = ca.descendant_concept_id
            WHERE ca.ancestor_concept_id IN ({ids(statin_ing)}) AND de.person_id IN ({ids(all_ids)})""")
st["d"] = pd.to_datetime(st.d)

def add_workup(coh, lo=0):
    """Flags for documented testing in [lo, 30] and [lo, 90] days; prior tests in [-180, lo-1]."""
    coh = coh.copy()
    x = tp.merge(coh[["person_id", "index_date"]], on="person_id")
    x["days"] = (x.d - x.index_date).dt.days
    prior_hi = min(-1, lo - 1)
    for k in TEST_PATTERNS:
        xk = x[x[k]]
        coh[f"{k}_prior"] = coh.person_id.isin(xk.loc[xk.days.between(-180, prior_hi), "person_id"])
        for w in (30, 90):
            coh[f"{k}_{w}"] = coh.person_id.isin(xk.loc[xk.days.between(lo, w), "person_id"])
    s = st.merge(coh[["person_id", "index_date"]], on="person_id")
    s["days"] = (s.d - s.index_date).dt.days
    coh["statin_prior"] = coh.person_id.isin(s.loc[s.days.between(-365, -1), "person_id"])
    coh["statin_90"] = coh.person_id.isin(s.loc[s.days.between(0, 90), "person_id"])
    coh["any_core_90"] = coh.echo_90 | coh.rhythm_90 | coh.carotid_90
    coh["newcore_90"] = coh.any_core_90
    coh["core_prior"] = coh.echo_prior | coh.rhythm_prior | coh.carotid_prior
    for w in (30, 90):
        coh[f"observable{w}"] = (coh.obs_end_day >= w) | coh.death_day.between(0, w)
    return coh

def eligible(coh, outcome, require_obs=True):
    """Window-specific observability; test-specific exclusions (see Methods)."""
    w = 30 if outcome.endswith("_30") else 90
    e = coh[f"observable{w}"] if require_obs else pd.Series(True, index=coh.index)
    k = outcome.split("_")[0]
    if k == "newcore":
        e = e & ~coh.core_prior
    if k in TEST_PATTERNS:
        e = e & ~coh[f"{k}_prior"]
    if k == "rhythm":
        e = e & ~coh["Prior AF"]
    if k == "statin":
        e = e & ~coh.statin_prior
    return e

coh_main = add_workup(coh_main)
coh_broad = add_workup(coh_broad)

OUTCOMES = [(f"{k}_{w}", f"{TEST_LABELS[k]}, {w} d") for k in TEST_PATTERNS for w in (30, 90)]
OUTCOMES += [("any_core_90", "Any documented core test (echo/rhythm/carotid), 90 d"),
             ("newcore_90", "Any core test, 90 d, among those with none in prior 180 d"),
             ("statin_90", "New statin, 90 d (statin-naive)")]

def rate_table(coh):
    rows = []
    for col, lab in OUTCOMES:
        r = {"measure": lab}
        w = 30 if col.endswith("_30") else 90
        for g in ["RAO", "TMVL"]:
            obs_n = int((coh[f"observable{w}"] & (coh.grp == g)).sum())
            d = coh[(coh.grp == g) & eligible(coh, col)]
            n, x = len(d), int(d[col].sum())
            diff = obs_n - n
            r[f"{g} eligible n"] = "suppressed" if 0 < diff <= MIN_CELL else n_str(n)
            r[f"{g} documented"] = pct_str(x, n)
        rows.append(r)
    return pd.DataFrame(rows)

save(rate_table(coh_main), "T2_documented_testing_rates", index=False)

## 7. Adjusted odds of documented testing, RAO vs TMVL (sequential models)

In [ ]:
CLIN_VARS = {c: c.replace(" ", "_") for c in BASE_CLIN}

def model_data(coh):
    d = coh[coh.sex.isin(["Female", "Male"])].copy()
    d["grp_rao"] = (d.grp == "RAO").astype(int)
    d["age10"] = (d.age - 65) / 10
    d["log_visits"] = np.log1p(d.prior_visit_days)
    for c, v in CLIN_VARS.items():
        d[v] = d[c].astype(int)
    return d

def rhs(d, level):
    t = ["grp_rao", "age10", "C(sex)", "C(race_eth, Treatment('White (NH)'))",
         "C(era, Treatment('2016 or earlier'))"]
    if level >= 2:
        t += [v for v in CLIN_VARS.values() if d[v].nunique() > 1]
    if level >= 3:
        t += ["log_visits"]
    if level >= 4:
        t += ["C(setting, Treatment('Outpatient'))"]
    return " + ".join(t)

def fit_logit(d, ycol, level, tag):
    d = d.assign(y=d[ycol].astype(int))
    rec = {"analysis": tag, "outcome": ycol, "model": f"M{level}", "n": len(d)}
    est = None
    try:
        with warnings.catch_warnings(record=True) as wl:
            warnings.simplefilter("always")
            m = smf.logit("y ~ " + rhs(d, level), data=d).fit(disp=0, maxiter=200)
        conv = (getattr(m, "mle_retvals", {}) or {}).get("converged", True) and \
               not any(issubclass(w.category, ConvergenceWarning) for w in wl)
        ci = m.conf_int()
        est = pd.DataFrame({"est": np.exp(m.params), "lo": np.exp(ci[0]), "hi": np.exp(ci[1])})
        rec.update(converged=bool(conv), error="")
        if not conv:
            est = None
    except Exception as e:
        rec.update(converged=False, error=f"{type(e).__name__}: {e}"[:120])
    FIT_LOG.append(rec)
    return est

MODEL_LABELS = {1: "M1 demographics + period", 2: "M2 + clinical history",
                3: "M3 + prior-year visits", 4: "M4 + care setting of diagnosis"}

def rao_vs_tmvl(coh, outcomes, tag, require_obs=True, levels=(1, 2, 3, 4)):
    rows = []
    for col, lab in outcomes:
        d = model_data(coh[eligible(coh, col, require_obs)])
        for lv in levels:
            e = fit_logit(d, col, lv, tag)
            if e is not None and "grp_rao" in e.index:
                r = e.loc["grp_rao"]
                rows.append({"outcome": lab, "model": MODEL_LABELS[lv], "OR": r.est, "lo": r.lo, "hi": r.hi})
            else:
                rows.append({"outcome": lab, "model": MODEL_LABELS[lv], "OR": np.nan, "lo": np.nan, "hi": np.nan})
    out = pd.DataFrame(rows)
    out["OR (95% CI), RAO vs TMVL"] = [fmt(a, b, c) for a, b, c in zip(out.OR, out.lo, out.hi)]
    return out

MAIN_OUT = [("any_core_90", "Any documented core test, 90 d"),
            ("newcore_90", "Any core test, 90 d (none in prior 180 d)"), ("echo_90", "Echocardiography, 90 d"),
            ("rhythm_90", "Rhythm monitoring, 90 d"), ("carotid_90", "Carotid/neck imaging, 90 d"),
            ("brain_90", "Brain imaging, 90 d"), ("statin_90", "New statin, 90 d")]
t3 = rao_vs_tmvl(coh_main, MAIN_OUT, "main")
t3.to_csv("results/T3_numeric.csv", index=False)
save(t3.pivot(index="outcome", columns="model", values="OR (95% CI), RAO vs TMVL")
       [list(MODEL_LABELS.values())], "T3_RAO_vs_TMVL_sequential_OR")

# Full M4 model for 'any core test' and 'new statin': all covariates
rows = []
for col, lab in [("any_core_90", "Any core test, 90 d"), ("statin_90", "New statin, 90 d")]:
    d = model_data(coh_main[eligible(coh_main, col)])
    e = fit_logit(d, col, 4, "main_full")
    if e is not None:
        for term, r in e.drop(index="Intercept").iterrows():
            rows.append({"outcome": lab, "term": term, "OR (95% CI)": fmt(r.est, r.lo, r.hi)})
save(pd.DataFrame(rows), "T4_full_model_M4", index=False)

## 8. Stroke after RAO and TMVL (secondary cases only)

In [ ]:
H = 1095

def stroke_set(coh, exclude_same_day=False, landmark=None):
    s = coh[~coh["Prior stroke"]].copy()
    cens = s.obs_end_day.clip(lower=0, upper=H)
    ts = s.stroke_day.where(s.stroke_day <= cens)
    td = s.death_day.where((s.death_day >= 0) & (s.death_day <= cens))
    s["time"] = np.fmin(np.fmin(ts.fillna(np.inf), td.fillna(np.inf)), cens) + 0.5
    s["status"] = np.select([ts.notna() & (ts <= td.fillna(np.inf)), td.notna()], [1, 2], 0)
    if exclude_same_day:
        s = s[~((s.status == 1) & (s.time <= 0.5))]
    if landmark:
        s = s[s.time > landmark + 0.5]
    return s

def cif_curve(t, s):
    """Aalen-Johansen cumulative incidence of status 1, death (status 2) competing."""
    df = pd.DataFrame({"t": t, "s": s})
    S, F, pts = 1.0, 0.0, [(0.0, 0.0)]
    for tt in np.sort(df.t[df.s > 0].unique()):
        n = (df.t >= tt).sum()
        at = df.t == tt
        F += S * (at & (df.s == 1)).sum() / n
        S *= 1 - (at & (df.s > 0)).sum() / n
        pts.append((tt, F))
    return pd.Series([p[1] for p in pts], index=[p[0] for p in pts])

def cif_table(coh, s, tag, ref=None):
    """ref = {group: n at risk in the main analysis}; rows whose population differs from it by 1-20
    people are fully suppressed, since the difference (and events in it) could be recovered."""
    rows = []
    for g in ["RAO", "TMVL"]:
        d = s[s.grp == g]
        excluded = int((coh.grp == g).sum()) - len(d)
        if ref is not None and 0 < abs(ref[g] - len(d)) <= MIN_CELL:
            rows.append({"analysis": tag, "group": g, "n at risk": "suppressed",
                         **{f"{p} d": "suppressed (population differs from main by ≤20)" for p in [30, 90, 365, 1095]}})
            continue
        cur = cif_curve(d.time.values, d.status.values)
        r = {"analysis": tag, "group": g,
             "n at risk": "suppressed" if 0 < excluded <= MIN_CELL else n_str(len(d))}
        for p in [30, 90, 365, 1095]:
            n_ev = int(((d.status == 1) & (d.time <= p + 0.5)).sum())
            val = cur[cur.index <= p + 0.5].iloc[-1]
            r[f"{p} d"] = f"{100 * val:.1f}%" if n_ev > MIN_CELL else "suppressed"
        rows.append(r)
    return pd.DataFrame(rows)

def fit_cox(tt, X, st, entry=None):
    last_err = None
    for method in ["newton", "bfgs", "lbfgs"]:
        try:
            with warnings.catch_warnings(record=True) as wl:
                warnings.simplefilter("always")
                m = PHReg(tt, X, status=st, entry=entry, ties="breslow").fit(method=method, maxiter=300)
            if not any(issubclass(w.category, ConvergenceWarning) for w in wl):
                return m, True
        except Exception as e:
            last_err = e
    if last_err:
        raise last_err
    return m, False

def cox_X(d, adjust):
    X = pd.DataFrame({"rao": (d.grp == "RAO").astype(float), "age10": (d.age - 65) / 10,
                      "female": (d.sex == "Female").astype(float)}, index=d.index)
    if adjust:
        for c in BASE_CLIN:
            X[c] = d[c].astype(float)
        rd = pd.get_dummies(d.race_eth, prefix="race").drop(columns="race_White (NH)", errors="ignore").astype(float)
        X = pd.concat([X, rd.loc[:, rd.sum() > MIN_CELL]], axis=1)
    return X.loc[:, X.nunique() > 1]

def cox_rao_vs_tmvl(s, horizon, adjust, tag, landmark=None):
    d = s[s.sex.isin(["Female", "Male"])].copy()
    X = cox_X(d, adjust)
    tt = np.minimum(d.time, horizon + 0.5).values
    stt = ((d.status == 1) & (d.time <= horizon + 0.5)).astype(int).values
    ev_r, ev_t = stt[(d.grp == "RAO").values].sum(), stt[(d.grp == "TMVL").values].sum()
    rec = {"analysis": tag, "outcome": f"stroke {horizon // 365} yr", "model": "adjusted" if adjust else "age/sex",
           "n": len(d)}
    res = {"analysis": tag, "follow-up": f"{horizon // 365} yr", "model": rec["model"],
           "HR": np.nan, "lo": np.nan, "hi": np.nan, "note": ""}
    if not (ev_r > MIN_CELL and ev_t > MIN_CELL):
        res["note"] = "suppressed (≤20 events in a group)"
        rec.update(converged=False, error="not fitted: small counts")
    else:
        try:
            entry = np.full(len(d), landmark + 0.5) if landmark else None
            m, conv = fit_cox(tt, X.values, stt, entry)
            b, (lo, hi) = float(m.params[0]), [float(v) for v in m.conf_int()[0]]
            rec.update(converged=bool(conv), error="")
            if conv:
                res.update(HR=np.exp(b), lo=np.exp(lo), hi=np.exp(hi))
            else:
                res["note"] = "did not converge"
        except Exception as e:
            rec.update(converged=False, error=f"{type(e).__name__}: {e}"[:120])
            res["note"] = "failed"
    FIT_LOG.append(rec)
    return res

s_main = stroke_set(coh_main)
REF_N = {g: int((s_main.grp == g).sum()) for g in ["RAO", "TMVL"]}
save(cif_table(coh_main, s_main, "main"), "T5_stroke_cumulative_incidence", index=False)

rows = [cox_rao_vs_tmvl(s_main, h, adj, "main") for adj in (False, True) for h in (365, 1095)]
t6 = pd.DataFrame(rows)
t6.to_csv("results/T6_numeric.csv", index=False)
t6["HR (95% CI), RAO vs TMVL"] = [fmt(a, b, c, 2) if pd.notna(a) else n for a, b, c, n in
                                  zip(t6.HR, t6.lo, t6.hi, t6.note)]
save(t6[["analysis", "model", "follow-up", "HR (95% CI), RAO vs TMVL"]], "T6_stroke_RAO_vs_TMVL_HR", index=False)

# ---- Proportional hazards: export for a formal survival::cox.zph test (companion R notebook) ----
# Individual-level file: stays inside the Workbench (not in results/). Same analysis population and
# covariates as the adjusted Cox models above; status 1 = stroke, 2 = death, 0 = censored; time in days + 0.5.
os.makedirs("internal", exist_ok=True)
_d = s_main[s_main.sex.isin(["Female", "Male"])].copy()
_X = cox_X(_d, True)
_X.columns = [re.sub(r"[^0-9A-Za-z]+", "_", c).strip("_") for c in _X.columns]
_ph = pd.concat([_d[["time", "status"]].reset_index(drop=True), _X.reset_index(drop=True)], axis=1)
_ph.to_csv("internal/stroke_ph_data_INTERNAL.csv", index=False)
print(f"Exported PH dataset for R: {len(_ph):,} rows, covariates: {', '.join(_X.columns)}")
print("Run rao_tmvl_ph_check_v2_3.ipynb (R kernel) next; it writes results/T6b_* files.")

# ---- Period-specific adjusted HRs (no constant-hazard-ratio assumption across periods) ----
def cox_period(s, a, b, tag):
    """Events in (a, b]. First period (a = 0): everyone, day-0 events included, no delayed entry.
    Later periods: people still event-free and under observation after day a, entering at a."""
    d = s[s.sex.isin(["Female", "Male"])].copy()
    if a > 0:
        d = d[d.time > a + 0.5]
    X = cox_X(d, True)
    tt = np.minimum(d.time, b + 0.5).values
    stt = ((d.status == 1) & (d.time <= b + 0.5)).astype(int).values
    ev_r, ev_t = stt[(d.grp == "RAO").values].sum(), stt[(d.grp == "TMVL").values].sum()
    rec = {"analysis": tag, "outcome": f"stroke days {a}-{b}", "model": "adjusted", "n": len(d)}
    out = {"period (days)": f"{a}-{b}", "HR (95% CI), RAO vs TMVL": "suppressed (≤20 events in a group)"}
    if ev_r > MIN_CELL and ev_t > MIN_CELL:
        try:
            entry = np.full(len(d), a + 0.5) if a > 0 else None
            m, conv = fit_cox(tt, X.values, stt, entry)
            b0, (lo, hi) = float(m.params[0]), [float(v) for v in m.conf_int()[0]]
            out["HR (95% CI), RAO vs TMVL"] = fmt(np.exp(b0), np.exp(lo), np.exp(hi)) if conv else "did not converge"
            rec.update(converged=bool(conv), error="")
        except Exception as e:
            out["HR (95% CI), RAO vs TMVL"] = "failed"
            rec.update(converged=False, error=f"{type(e).__name__}")
    else:
        rec.update(converged=False, error="not fitted: small counts")
    FIT_LOG.append(rec)
    return out

_t6c = pd.DataFrame([cox_period(s_main, a, b, "period") for a, b in [(0, 30), (30, 365), (365, 1095), (30, 1095)]])
_t6c["note"] = ["includes day-0 events", "", "",
                "same population and entry as the 30-day landmark analysis (S4); not independent confirmation"]
save(_t6c, "T6c_stroke_HR_by_period", index=False)

## 9. Sensitivity analyses

In [ ]:
ANY = [("any_core_90", "Any of echo/rhythm/carotid, 90 d")]
STATIN = [("statin_90", "New statin, 90 d")]
coh_win = add_workup(coh_main, lo=-14)
coh_conf = coh_main[coh_main.confirmed]
coh_noboundary = coh_main[~coh_main.boundary_year]

sens = []
def add_sens(label, df):
    df = df.copy(); df.insert(0, "analysis", label); sens.append(df)

add_sens("Main", rao_vs_tmvl(coh_main, ANY, "s_main", levels=(1, 4)))
add_sens("Window from day -14", rao_vs_tmvl(coh_win, ANY, "s_window", levels=(1, 4)))
add_sens("No 90-day observability requirement", rao_vs_tmvl(coh_main, ANY, "s_allobs", require_obs=False, levels=(1, 4)))
add_sens("Selected subgroup: RAO recorded on ≥2 dates (uses later records)", rao_vs_tmvl(coh_conf, ANY, "s_confirmed", levels=(1, 4)))
add_sens("Broad RAO definition", rao_vs_tmvl(coh_broad, ANY, "s_broad", levels=(1, 4)))
add_sens("Excluding shifted boundary years", rao_vs_tmvl(coh_noboundary, ANY, "s_boundary", levels=(1, 4)))
s1 = pd.concat(sens, ignore_index=True)
s1.to_csv("results/S1_numeric.csv", index=False)
save(s1.pivot(index="analysis", columns="model", values="OR (95% CI), RAO vs TMVL"), "S1_testing_OR_sensitivity")

# Statin-initiation period effect, main vs excluding shifted boundary years
rows = []
for label, c in [("Main", coh_main), ("Excluding shifted boundary years", coh_noboundary)]:
    d = model_data(c[eligible(c, "statin_90")])
    e = fit_logit(d, "statin_90", 4, "s_era_" + label)
    if e is not None:
        for term in [t for t in e.index if "era" in t]:
            rows.append({"analysis": label, "period vs 2016 or earlier": term.split("T.")[-1].rstrip("]"),
                         "OR (95% CI), new statin": fmt(*e.loc[term, ["est", "lo", "hi"]])})
save(pd.DataFrame(rows), "S2_statin_period_sensitivity", index=False)

# Stroke: cumulative incidence and RAO-vs-TMVL HR under alternative definitions
stroke_sets = {"Main": (coh_main, stroke_set(coh_main), None),
               "Excluding same-day strokes": (coh_main, stroke_set(coh_main, exclude_same_day=True), None),
               "Landmark: after day 30": (coh_main, stroke_set(coh_main, landmark=30), 30),
               "Selected subgroup: RAO on ≥2 dates": (coh_conf, stroke_set(coh_conf), None),
               "Broad RAO definition": (coh_broad, stroke_set(coh_broad), None)}
save(pd.concat([cif_table(c, s, k, None if k == "Main" else REF_N) for k, (c, s, lm) in stroke_sets.items()],
               ignore_index=True),
     "S3_stroke_incidence_sensitivity", index=False)
rows = [cox_rao_vs_tmvl(s, h, True, k, landmark=lm) for k, (c, s, lm) in stroke_sets.items() for h in (365, 1095)]
s4 = pd.DataFrame(rows)
s4.to_csv("results/S4_numeric.csv", index=False)
s4["adjusted HR (95% CI), RAO vs TMVL"] = [fmt(a, b, c) if pd.notna(a) else n
                                           for a, b, c, n in zip(s4.HR, s4.lo, s4.hi, s4.note)]
save(s4[["analysis", "follow-up", "adjusted HR (95% CI), RAO vs TMVL"]], "S4_stroke_HR_sensitivity", index=False)

## 10. Model fitting log and outputs

In [ ]:
log = pd.DataFrame(FIT_LOG)
summary = log.groupby("analysis").agg(attempted=("converged", "size"), successful=("converged", "sum"))
save(summary, "S5_model_fit_summary")
failed = log[~log.converged.astype(bool)]
if len(failed):
    print("Models not estimated:")
    display(failed[["analysis", "outcome", "model", "error"]])
log.drop(columns=["n"]).to_csv("results/S5_model_fit_log.csv", index=False)

print("\nSaved result tables:")
for f in sorted(os.listdir("results")):
    print("  results/" + f)
import hashlib, shutil
shutil.copy(PROC_FILE, "results/definitions_procedure_concepts_used.csv")
print("\nProcedure definitions used: MD5", hashlib.md5(open(PROC_FILE, "rb").read()).hexdigest())
print("\nNext: File → Save, then File → Save and Export Notebook As → HTML.")
print("Then run rao_tmvl_ph_check_v2_3.ipynb (R kernel) for the formal proportional-hazards test.")